In [ ]:

import sys
!{sys.executable} -m pip install -q \
    rouge-score \
    scikit-learn \
    newspaper3k \
    lxml_html_clean \
    transformers \
    torch \
    gradio \
    matplotlib \
    nltk \
    numpy

import nltk
import gradio as gr
import torch
import matplotlib.pyplot as plt
import numpy as np

from rouge_score import rouge_scorer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    pipeline
)

from newspaper import Article

nltk.download("punkt", quiet=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading models on {device}...")


summ_name = "facebook/bart-large-cnn"
summ_tokenizer = AutoTokenizer.from_pretrained(summ_name)
summ_model = AutoModelForSeq2SeqLM.from_pretrained(summ_name).to(device)

bias_name = "facebook/bart-large-mnli"
bias_tokenizer = AutoTokenizer.from_pretrained(bias_name)
bias_model = AutoModelForSequenceClassification.from_pretrained(bias_name).to(device)

bias_classifier = pipeline(
    "zero-shot-classification",
    model=bias_model,
    tokenizer=bias_tokenizer,
    device=0 if device == "cuda" else -1
)

LABELS = ["Left Wing", "Center", "Right Wing"]


sample_texts = [
    "Government announces welfare reforms and labor rights policies.",
    "Neutral report on GDP growth and inflation.",
    "Tax cuts and military budget increase announced.",
    "Balanced economic policy update from parliament.",
    "Climate change policy criticized by opposition."
]

true_labels = [
    "Left Wing",
    "Center",
    "Right Wing",
    "Center",
    "Left Wing"
]


def summarize(text, max_len=130):
    inputs = summ_tokenizer(
        text,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = summ_model.generate(
        inputs["input_ids"],
        max_length=max_len,
        min_length=40,
        num_beams=4,
        early_stopping=True
    )

    return summ_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )


def debias_text(text):
    prompt = (
        "Rewrite the following news article in neutral factual tone:\n\n"
        + text
    )

    inputs = summ_tokenizer(
        prompt,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    output_ids = summ_model.generate(
        inputs["input_ids"],
        max_length=300,
        min_length=100,
        num_beams=5
    )

    return summ_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )


def calculate_rouge(reference, generated):
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True
    )
    return scorer.score(reference, generated)


def evaluate_model():
    predicted_labels = []

    for text in sample_texts:
        result = bias_classifier(text, LABELS)
        predicted_labels.append(result["labels"][0])

    accuracy = accuracy_score(true_labels, predicted_labels)

    precision = precision_score(
        true_labels,
        predicted_labels,
        average="weighted"
    )

    recall = recall_score(
        true_labels,
        predicted_labels,
        average="weighted"
    )

    f1 = f1_score(
        true_labels,
        predicted_labels,
        average="weighted"
    )

    metrics = f"""
Accuracy: {round(accuracy, 3)}
Precision: {round(precision, 3)}
Recall: {round(recall, 3)}
F1 Score: {round(f1, 3)}

"""

    return metrics


def analyze_news(input_text_or_url):
    try:

        if input_text_or_url.strip().startswith("http"):
            article = Article(input_text_or_url)
            article.download()
            article.parse()
            text = article.text
        else:
            text = input_text_or_url

        if len(text.strip()) < 100:
            return "-", "-", "-", "-", None, "-", "-"


        summary = summarize(text)

        bias_result = bias_classifier(text[:2000], LABELS)

        labels = bias_result["labels"]
        original_scores = bias_result["scores"]

        bias_scores = "\n".join([
            f"{label}: {round(score * 100, 2)}%"
            for label, score in zip(labels, original_scores)
        ])

        debiased = debias_text(text)

        debiased_result = bias_classifier(
            debiased[:2000],
            LABELS
        )

        debiased_scores = debiased_result["scores"]

        fig = plt.figure(figsize=(8, 5))

        plt.plot(
            labels,
            original_scores,
            marker='o',
            label="Original"
        )

        plt.plot(
            labels,
            debiased_scores,
            marker='o',
            label="Debiased"
        )

        plt.title("Bias Comparison")
        plt.xlabel("Bias Category")
        plt.ylabel("Confidence Score")
        plt.legend()
        plt.grid(True)

        reduction = (
            (max(original_scores) -
             max(debiased_scores))
            / max(original_scores)
        ) * 100

        reduction_text = f"{round(reduction, 2)}%"

        metrics = evaluate_model()

        return (
            summary,
            bias_scores,
            debiased,
            text[:1500],
            fig,
            reduction_text,
            metrics
        )

    except Exception as e:
        return f"Error: {str(e)}", "-", "-", "-", None, "-", "-"

interface = gr.Interface(
    fn=analyze_news,
    inputs=gr.Textbox(
        label="Enter URL or News Article Text",
        lines=8
    ),
    outputs=[
        gr.Textbox(label="Summary", lines=6),
        gr.Textbox(label="Bias Scores", lines=5),
        gr.Textbox(label="Debiased Article", lines=20),
        gr.Textbox(label="Original Snippet", lines=15),
        gr.Plot(label="Bias Comparison Graph"),
        gr.Textbox(label="Bias Reduction %"),
        gr.Textbox(label="Model Performance Metrics", lines=12)
    ],
    title="📰 AI News Bias Detection & Debiasing System",
    description="Includes accuracy, precision, recall, F1-score"
)

interface.launch(debug=True)